# How Tensors Travel Around in the System

A tensor is cheap to *pass around* and expensive to *copy*. This notebook is about telling the two
apart: which boundaries a tensor can cross by handing over a pointer and a few integers, and which
boundaries force its bytes to be moved.

The short answer for PyTorch multiprocessing, which the notebook then takes apart and measures:

> PyTorch does not normally copy tensor bytes through a multiprocessing pipe. It puts the tensor's
> CPU `Storage` into shared memory, sends a small handle plus tensor metadata through the queue, and
> maps that same storage into the receiving process.

"Zero-copy" needs qualification, though, and every one of these claims is measured below:

* If the tensor is already in shared memory, the inter-process transfer is zero-copy for its payload.
* If it is an ordinary CPU tensor, the first send copies its entire storage once into shared memory.
* `DataLoader` avoids this extra first-send copy by constructing its batch directly in shared memory.
* `pin_memory=True` adds another CPU copy into pinned memory.
* Moving the batch to CUDA adds a host-to-device copy.

**How to read this notebook.** Each section follows the same four beats:

1. **Concept** — what problem the boundary poses and what "zero-copy" would even mean there.
2. **Mechanism** — the actual call chain, with pointers into the PyTorch source tree.
3. **Experiment** — a cell that proves (or disproves) the claim with `assert`s and measurements.
4. **Gotchas** — the failure modes, and something to try yourself.

Source pointers use `file:line` from the *installed* PyTorch version printed by the next cell; line
numbers drift between releases, so the `show_source()` helper is there to read the real thing
locally. Two sections are deliberately left open (language boundary, node-to-node over the network).

In [1]:
# Helpers used throughout the notebook. Nothing here is magic: describe() just prints the two
# halves of a tensor (metadata vs storage) side by side, and show_source() reads the installed
# PyTorch implementation so the source pointers in this notebook stay checkable.
import gc
import inspect
import json
import os
import pathlib
import subprocess
import sys
import tempfile
import time
import warnings

import numpy as np
import torch
import torch.multiprocessing as mp

MB = 1024 * 1024


def describe(t, name="tensor"):
    """Print the metadata half and the storage half of a tensor separately."""
    st = t.untyped_storage()
    print(f"{name}:")
    print(f"  metadata : shape={tuple(t.shape)} stride={t.stride()} "
          f"offset={t.storage_offset()} dtype={t.dtype} device={t.device}")
    print(f"  storage  : data_ptr=0x{t.data_ptr():x} nbytes={st.nbytes()} "
          f"is_shared={t.is_shared()} "
          f"is_pinned={t.is_pinned() if t.device.type == 'cpu' else 'n/a'}")


def same_storage(a, b):
    """True when two tensors point at the same StorageImpl bytes."""
    return a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr()


def aliases_numpy(t, a):
    """True when a tensor and an ndarray start at the same byte."""
    return t.data_ptr() == a.__array_interface__["data"][0]


def shm_used():
    """Bytes currently allocated in /dev/shm (tmpfs) -- our meter for IPC copies."""
    s = os.statvfs("/dev/shm")
    return (s.f_blocks - s.f_bfree) * s.f_frsize


def rss_mib():
    """Resident set size of this process in MiB, from /proc/self/statm."""
    fields = pathlib.Path("/proc/self/statm").read_text().split()
    return int(fields[1]) * os.sysconf("SC_PAGE_SIZE") / MB


def _strip_docstring(lines):
    out, state = [], "head"
    for line in lines:
        s = line.strip()
        if state == "head":
            out.append(line)
            if s.endswith(":") and (s.startswith("def ") or s.startswith(") ->") or s == "):"):
                state = "maybe_doc"
        elif state == "maybe_doc":
            s = s.lstrip("rbuf") if s[:1].lower() in "rbuf" else s
            if not s:
                out.append(line)
            elif s.startswith(('"""', "'''")):
                quote = s[:3]
                state = "body" if (len(s) > 3 and s.endswith(quote)) else "in_doc"
            else:
                out.append(line)
                state = "body"
        elif state == "in_doc":
            if s.endswith(('"""', "'''")):
                state = "body"
        else:
            out.append(line)
    return out


def show_source(obj, head=None, keep_doc=False):
    """Print `file:line` plus the installed PyTorch implementation of a Python-level object."""
    try:
        file = inspect.getsourcefile(obj)
        src, line = inspect.getsourcelines(obj)
    except (TypeError, OSError) as e:      # C-implemented builtins have no Python source
        print(f"<no Python source for {obj!r}: {e}>")
        return
    root = os.path.dirname(os.path.dirname(torch.__file__))
    print(f"# {os.path.relpath(file, root)}:{line}   {getattr(obj, '__qualname__', obj)}")
    if not keep_doc:
        src = _strip_docstring(src)
    if head is not None and len(src) > head:
        src = src[:head] + ["    ...\n"]
    print("".join(src).rstrip())


def show_lines(module, start, end):
    """Print an exact line range of a module's file, with line numbers."""
    file = inspect.getsourcefile(module)
    root = os.path.dirname(os.path.dirname(torch.__file__))
    text = pathlib.Path(file).read_text().splitlines()
    print(f"# {os.path.relpath(file, root)}:{start}-{end}")
    for i in range(start, end + 1):
        print(f"{i:5d}  {text[i - 1]}")


# NOTE: we deliberately do NOT touch a CUDA device yet. is_available()/device_count() do not create
# a CUDA context, but anything like get_device_name() does -- and a process that has initialized
# CUDA cannot safely fork(), which the process-boundary experiments below rely on.
_shm = os.statvfs("/dev/shm")
print(f"torch {torch.__version__} | numpy {np.__version__} | python {sys.version.split()[0]}")
print(f"sharing strategy : {mp.get_sharing_strategy()} "
      f"(available: {sorted(mp.get_all_sharing_strategies())})")
print(f"cuda available   : {torch.cuda.is_available()} | device_count={torch.cuda.device_count()}")
print(f"/dev/shm         : {shm_used() / MB:.1f} MiB used of "
      f"{_shm.f_blocks * _shm.f_frsize / MB / 1024:.0f} GiB")
print(f"torch source at  : {os.path.dirname(torch.__file__)}")

torch 2.13.0+cu130 | numpy 2.5.1 | python 3.12.3
sharing strategy : file_descriptor (available: ['file_descriptor', 'file_system'])
cuda available   : True | device_count=1
/dev/shm         : 0.2 MiB used of 60 GiB
torch source at  : /home/huangruoyu/workspace/kernel-lab/.venv/lib/python3.12/site-packages/torch


## Tensor Transfer Zero-Copy

Tensor is consists of data storage and metadata (e.g. shape, strides, dtype, device, storage offsets). 

When transforming tensors, the storage usually does not change, only view metadata changes. 
When passing tensor between programming boundary, library boundary, process boundary, we also want pass tensor with zero-copy of underlying storage, only copy metadata.


A dense PyTorch tensor is conceptually split into:

```text
TensorImpl
  ├── shape
  ├── strides
  ├── storage_offset
  ├── dtype / device
  └── reference to StorageImpl
                         └── DataPtr → actual bytes
```

Multiple tensors can reference the same storage:

```python
x = torch.arange(100)
y = x[20:30]             # same storage, different offset/shape
```




**Where this lives in the source.** The split above is not a teaching abstraction, it is the actual
class layout:

| Concept | Type | Source |
| --- | --- | --- |
| shape / strides / offset / dtype / device | `TensorImpl` | [`c10/core/TensorImpl.h`](https://github.com/pytorch/pytorch/blob/main/c10/core/TensorImpl.h) |
| the bytes, their size, their allocator | `StorageImpl` | [`c10/core/StorageImpl.h`](https://github.com/pytorch/pytorch/blob/main/c10/core/StorageImpl.h) |
| pointer + deleter that frees it | `DataPtr` | [`c10/core/Allocator.h`](https://github.com/pytorch/pytorch/blob/main/c10/core/Allocator.h) |

A **view** is a new `TensorImpl` that bumps the refcount on an existing `StorageImpl`. So "does this
operation copy?" is the same question as "did a new `StorageImpl` appear?", and
`same_storage(a, b)` in the helper cell answers it by comparing
`untyped_storage().data_ptr()`.

Note the asymmetry that the next cell makes visible: `t.data_ptr()` is
*storage base + storage_offset × itemsize*, while `t.untyped_storage().data_ptr()` is the storage
base. Two views of one storage have different `data_ptr()` but the same storage pointer.

In [2]:
# Predict before running: what is y.untyped_storage().nbytes() -- 40 bytes, or 400?
x = torch.arange(100, dtype=torch.float32)
y = x[20:30]

describe(x, "x = torch.arange(100)")
describe(y, "y = x[20:30]")
print(f"same storage        : {same_storage(x, y)}")
print(f"data_ptr difference : {y.data_ptr() - x.data_ptr()} bytes "
      f"= storage_offset {y.storage_offset()} x {x.element_size()} bytes/element")
print(f"storage nbytes      : x={x.untyped_storage().nbytes()}  y={y.untyped_storage().nbytes()}"
      f"   <- y sees 10 elements but owns all 400 bytes")

print("\n-- metadata-only operations: no new storage --")
for label, t in [("x.view(10, 10)", x.view(10, 10)),
                 ("x.view(10, 10).t()", x.view(10, 10).t()),
                 ("x[::2]", x[::2]),
                 ("x.unsqueeze(0).expand(4, 100)", x.unsqueeze(0).expand(4, 100))]:
    print(f"  {label:32s} same_storage={str(same_storage(x, t)):5s} "
          f"shape={tuple(t.shape)} stride={t.stride()}")

print("\n-- operations that must allocate --")
for label, t in [("x.view(10, 10).t().contiguous()", x.view(10, 10).t().contiguous()),
                 ("x.clone()", x.clone()),
                 ("x.to(torch.float64)", x.to(torch.float64)),
                 ("x + 0", x + 0)]:
    print(f"  {label:32s} same_storage={str(same_storage(x, t)):5s} "
          f"new data_ptr=0x{t.data_ptr():x}")

y[0] = 999.0
assert x[20].item() == 999.0
print(f"\nwriting y[0] = 999 is visible as x[20] = {x[20].item()}  (one storage, two views)")

x = torch.arange(100):
  metadata : shape=(100,) stride=(1,) offset=0 dtype=torch.float32 device=cpu
  storage  : data_ptr=0x3ea580102c0 nbytes=400 is_shared=False is_pinned=False
y = x[20:30]:
  metadata : shape=(10,) stride=(1,) offset=20 dtype=torch.float32 device=cpu
  storage  : data_ptr=0x3ea58010310 nbytes=400 is_shared=False is_pinned=False
same storage        : True
data_ptr difference : 80 bytes = storage_offset 20 x 4 bytes/element
storage nbytes      : x=400  y=400   <- y sees 10 elements but owns all 400 bytes

-- metadata-only operations: no new storage --
  x.view(10, 10)                   same_storage=True  shape=(10, 10) stride=(10, 1)
  x.view(10, 10).t()               same_storage=True  shape=(10, 10) stride=(1, 10)
  x[::2]                           same_storage=True  shape=(50,) stride=(2,)
  x.unsqueeze(0).expand(4, 100)    same_storage=True  shape=(4, 100) stride=(0, 1)

-- operations that must allocate --
  x.view(10, 10).t().contiguous()  same_storage=False new

**Gotcha: a view keeps the whole storage alive.** `y` above is 40 bytes of interest sitting on a
400-byte storage, and `x` can be garbage collected while `y` keeps all 400 bytes allocated. On a
40-byte storage nobody cares. The same arithmetic with a 64 MiB storage is how a `[:10]` slice ends
up copying 64 MiB into `/dev/shm` -- measured in the process-boundary section below.

**Try it:** `x.view(10, 10).t()` is a view, but `x.view(10, 10).t().view(-1)` raises. Why? (Strides
`(1, 10)` cannot be expressed as a single contiguous run, so `view` refuses and `reshape` copies.)

### Transfer Between Library Boundary

The data storage is typically what people think of as arrays in C or Fortran, a contiguous (and fixed) block of memory containing fixed-sized data items. 

**NumPy → PyTorch**

`torch.from_numpy()` does not allocate or copy the data. Internally PyTorch:

1. Reads NumPy’s pointer, sizes, byte strides, and dtype.
2. Converts byte strides into element strides.
3. Calls at::from_blob() around PyArray_DATA(array).
4. Increments the NumPy object’s reference count.
5. Installs a storage deleter that decrements the NumPy reference when the tensor dies.

The implementation is in `tensor_numpy.cpp`; the behavior is documented by `torch.from_numpy`.


In [3]:
import numpy as np
import torch
a = np.arange(10, dtype=np.float32)
t = torch.from_numpy(a)

t[0] = 99
assert a[0] == 99

In [4]:
# Which numpy -> torch constructors actually alias, and which quietly copy?
a = np.arange(10, dtype=np.float32)

print(f"{'constructor':36s} {'aliases numpy?':>15s}   note")
for label, t, note in [
    ("torch.from_numpy(a)", torch.from_numpy(a), "zero-copy view over PyArray_DATA(a)"),
    ("torch.as_tensor(a)", torch.as_tensor(a), "same fast path when dtype/device match"),
    ("torch.as_tensor(a, dtype=float64)", torch.as_tensor(a, dtype=torch.float64), "dtype conversion -> copy"),
    ("torch.tensor(a)", torch.tensor(a), "torch.tensor always copies"),
]:
    print(f"{label:36s} {str(aliases_numpy(t, a)):>15s}   {note}")

print("\n-- inputs the zero-copy path refuses --")
try:
    torch.from_numpy(a[::-1])
except ValueError as e:
    print(f"negative strides : ValueError: {str(e)[:78]}...")
print(f"  workaround     : torch.from_numpy(a[::-1].copy()) -> "
      f"{torch.from_numpy(a[::-1].copy())[:3].tolist()} (now it is a copy)")

try:
    torch.from_numpy(np.arange(4, dtype='>f4'))
except ValueError as e:
    print(f"byte order       : ValueError: {str(e)[:78]}...")

ro = np.arange(4, dtype=np.float32)
ro.flags.writeable = False
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    t_ro = torch.from_numpy(ro)
    print(f"read-only array  : {w[0].category.__name__}: {str(w[0].message)[:70]}...")

t = torch.from_numpy(a)
try:
    t.resize_(20)
except RuntimeError as e:
    print(f"resize_ on blob  : RuntimeError: {str(e)[:78]}")

constructor                           aliases numpy?   note
torch.from_numpy(a)                             True   zero-copy view over PyArray_DATA(a)
torch.as_tensor(a)                              True   same fast path when dtype/device match
torch.as_tensor(a, dtype=float64)              False   dtype conversion -> copy
torch.tensor(a)                                False   torch.tensor always copies

-- inputs the zero-copy path refuses --
negative strides : ValueError: At least one stride in the given numpy array is negative, and tensors with neg...
  workaround     : torch.from_numpy(a[::-1].copy()) -> [9.0, 8.0, 7.0] (now it is a copy)
byte order       : ValueError: given numpy array has byte order different from the native byte order. Convers...
read-only array  : UserWarning: The given NumPy array is not writable, and PyTorch does not support no...
resize_ on blob  : RuntimeError: Trying to resize storage that is not resizable


Zero-copy from numpy to torch requires:

* CPU memory only.
* Supported dtype must have a direct PyTorch equivalent.
* Native byte order is required.
* Negative NumPy strides are rejected; use `array.copy()`.
* The resulting tensor is not resizable.
* A read-only NumPy array produces a warning; writing through the tensor is undefined behavior.


`torch.as_tensor(a)` uses the same zero-copy path when dtype and device are compatible. 
However, a dtype conversion or `device="cuda"` requires a copy. `torch.tensor(a)` always copies.

**PyTorch → NumPy**

PyTorch creates an ndarray using:

* `prepared_tensor.data_ptr()` as the array data pointer,
* tensor sizes,
* tensor strides converted from elements to bytes,
* a Python tensor wrapper as `ndarray.base`.

The base reference keeps the tensor alive, and the storage is marked non-resizable. The implementation is also in [`tensor_numpy.cpp`](https://github.com/pytorch/pytorch/blob/main/torch/csrc/utils/tensor_numpy.cpp).

In [5]:
t = torch.arange(10, dtype=torch.float32)
a = t.numpy()

a[0] = 99
assert t[0] == 99

In [6]:
# The ndarray does not own the bytes -- the tensor does, and `base` is what keeps it alive.
t = torch.arange(10, dtype=torch.float32)
arr = t.numpy()
print(f"type(arr.base)      : {type(arr.base).__name__}   (a torch.Tensor wrapper, not the ndarray's own buffer)")
print(f"arr.flags.owndata   : {arr.flags.owndata}")
print(f"same bytes as t      : {arr.base.data_ptr() == t.data_ptr()}")
del t
gc.collect()
print(f"after `del t`, arr is still valid: sum={arr.sum()}   (arr.base holds the last reference)")

print("\n-- when .numpy() refuses, and what force=True does instead --")
g = torch.ones(3, requires_grad=True)
try:
    g.numpy()
except RuntimeError as e:
    print(f"requires_grad : RuntimeError: {str(e)[:80]}")
print(f"  detach()    : {g.detach().numpy().tolist()} (still zero-copy: "
      f"{aliases_numpy(g.detach(), g.detach().numpy())})")

c = torch.tensor([1 + 2j, 3 + 4j]).conj()
try:
    c.numpy()
except RuntimeError as e:
    print(f"conjugate bit : RuntimeError: {str(e)[:80]}")
forced = c.numpy(force=True)
print(f"  force=True  : {forced} -- aliases={aliases_numpy(c, forced)} (it materialized a copy)")

nc = torch.arange(12).reshape(3, 4).t()
print(f"\nnon-contiguous tensors are fine: strides {nc.stride()} become byte strides "
      f"{nc.numpy().strides}, aliases={aliases_numpy(nc, nc.numpy())}")

type(arr.base)      : Tensor   (a torch.Tensor wrapper, not the ndarray's own buffer)
arr.flags.owndata   : False
same bytes as t      : True
after `del t`, arr is still valid: sum=45.0   (arr.base holds the last reference)

-- when .numpy() refuses, and what force=True does instead --
requires_grad : RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() ins
  detach()    : [1.0, 1.0, 1.0] (still zero-copy: True)
conjugate bit : RuntimeError: Can't call numpy() on Tensor that has conjugate bit set. Use tensor.resolve_conj
  force=True  : [1.-2.j 3.-4.j] -- aliases=False (it materialized a copy)

non-contiguous tensors are fine: strides (1, 4) become byte strides (8, 32), aliases=True


Zero-copy `tensor.numpy()` requires a compatible:

* CPU tensor,
* strided layout,
* NumPy-supported dtype,
* no unresolved conjugate or negative bit,
* no active autograd requirement unless detached.

`tensor.numpy(force=True)` is effectively: `tensor.detach().cpu().resolve_conj().resolve_neg().numpy()`

It may therefore copy. See [`Tensor.numpy`](https://docs.pytorch.org/docs/2.13/generated/torch.Tensor.numpy.html).

**Mechanism.** Both directions are ~200 lines of C++ in
[`torch/csrc/utils/tensor_numpy.cpp`](https://github.com/pytorch/pytorch/blob/main/torch/csrc/utils/tensor_numpy.cpp),
and the rules above fall straight out of them:

```text
torch.from_numpy(a)                          tensor.numpy()
  tensor_from_numpy()                          tensor_to_numpy()
    PyArray_DATA(a)          -> data ptr         prepared_tensor.data_ptr() -> data ptr
    PyArray_DIMS / STRIDES   -> sizes,           sizes                      -> dims
      byte strides / itemsize -> element strides  element strides x itemsize -> byte strides
    Py_INCREF(a)                                 py_tensor                  -> ndarray.base
    at::from_blob(ptr, sizes, strides, deleter)  storage marked non-resizable
      deleter: Py_DECREF(a) when the tensor dies
```

| Behavior you observed | Why |
| --- | --- |
| negative strides rejected | element strides must be non-negative in `TensorImpl` |
| non-native byte order rejected | there is no dtype to map it to; the bytes would be misread |
| `resize_` fails | `from_blob` installs a non-resizable storage -- PyTorch does not own the allocation |
| read-only array only warns | the tensor has no writability flag to honor, so writing is UB |
| `arr.base` is a `torch.Tensor` | it is the reference that keeps the storage alive after `del t` |

`at::from_blob` is the general escape hatch here: *any* pointer you can describe with sizes, strides
and a dtype can become a tensor without a copy. Shared memory (next section) and mmap'd files
(device-boundary section) are the same trick with a different allocator behind the pointer.

### Transfer Between Process Boundary

For multiprocessing, PyTorch primarily shares the `Storage`, while serializing the tensor’s small metadata:

* tensor class
* dtype
* shape
* strides
* storage offset
* `requires_grad`
* shared-memory handle for the storage

`tensor.share_memory_()` Moves the underlying storage to shared memory. This is a no-op if the underlying storage is already in shared memory. Tensors in shared memory cannot be resized.

**The first send is not free.** `share_memory_()` (and the queue reducer that calls it for you) has
to get the bytes *into* a shared mapping, and an ordinary CPU allocation is not one. The C++ side,
[`THPStorage_shareFd`](https://github.com/pytorch/pytorch/blob/main/torch/csrc/StorageSharing.cpp),
does this:

1. Is the storage already managed by `MapAllocator`? Then just return its file descriptor.
2. Otherwise: allocate a new POSIX shared-memory storage of the same size,
3. `storage_copy(new_storage, storage)` -- **one full copy of the payload**,
4. replace the original storage's `DataPtr` and allocator with the shared allocation,
5. return the shared-memory FD.

So the cost depends entirely on where the storage came from:

| Sender storage | Payload copy during first send | Later sends |
| --- | ---: | ---: |
| Ordinary CPU allocation | one full storage copy into SHM | zero-copy |
| `tensor.share_memory_()` already called | none | zero-copy |
| DataLoader batch built directly in SHM | none during queue serialization | zero-copy |
| CUDA storage | uses CUDA IPC, not `/dev/shm` | no device-to-device payload copy |

Step 4 is the subtle one: the storage is *replaced in place*, so `data_ptr()` changes and anything
that was aliasing those bytes (a NumPy array, say) is left behind pointing at the old allocation.

> **Warning.** The docs say shared storages are not resizable, but in the installed 2.13.0 build
> `storage.resizable()` still reports `True` and *growing* a shared storage segfaults the process
> rather than raising. Verified while writing this notebook, which is why there is no live cell for
> it -- treat a shared tensor's size as frozen.

In [7]:
# Predict: after share_memory_(), does data_ptr() stay the same?
t = torch.arange(8 * MB // 4, dtype=torch.float32)          # 8 MiB of ordinary CPU memory

before = shm_used()
print(f"before : is_shared={t.is_shared()} data_ptr=0x{t.data_ptr():x}")
t0 = time.perf_counter()
t.share_memory_()
dt = (time.perf_counter() - t0) * 1e3
grew = (shm_used() - before) / MB
print(f"after  : is_shared={t.is_shared()} data_ptr=0x{t.data_ptr():x}   <- new storage, new address")
print(f"/dev/shm grew by {grew:.1f} MiB in {dt:.2f} ms   <- the one-time copy into shared memory")

t1 = time.perf_counter()
t.share_memory_()
print(f"calling it again  : {(time.perf_counter() - t1) * 1e6:.0f} us, "
      f"/dev/shm delta {(shm_used() - before) / MB - grew:+.1f} MiB   <- no-op once shared")

del t
gc.collect()
print(f"after `del t`     : /dev/shm back to {shm_used() / MB:.1f} MiB "
      f"(the mapping is freed when the last reference goes)")

before : is_shared=False data_ptr=0x3ea58060000
after  : is_shared=True data_ptr=0xfae105750000   <- new storage, new address
/dev/shm grew by 8.0 MiB in 1.95 ms   <- the one-time copy into shared memory
calling it again  : 45 us, /dev/shm delta +0.0 MiB   <- no-op once shared
after `del t`     : /dev/shm back to 0.2 MiB (the mapping is freed when the last reference goes)


**Gotcha: sharing a view shares the whole storage.** The reducer moves the *storage*, not the ten
elements you can see through the view:

```python
large = torch.empty(16_777_216)   # 64 MiB
small = large[:10]                # 40 bytes of interest...
queue.put(small)                  # ...64 MiB into /dev/shm
```

`small.clone()` gives it a compact storage of its own and the problem disappears. The next cell
measures both, and then shows the related aliasing trap: because `share_memory_()` *replaces* the
storage, a tensor built with `torch.from_numpy()` stops aliasing its NumPy array the moment you
share it.

In [8]:
# The view trap, measured in /dev/shm.
large = torch.empty(64 * MB // 4, dtype=torch.float32)      # 64 MiB storage
small = large[:10]                                          # 40 bytes visible
print(f"small.numel()={small.numel()} but "
      f"small.untyped_storage().nbytes()={small.untyped_storage().nbytes() / MB:.0f} MiB")

base = shm_used()
small.share_memory_()
print(f"small.share_memory_()          -> /dev/shm {(shm_used() - base) / MB:+.1f} MiB")

large2 = torch.empty(64 * MB // 4, dtype=torch.float32)
compact = large2[:10].clone()
base = shm_used()
compact.share_memory_()
print(f"large2[:10].clone() then share  -> /dev/shm {(shm_used() - base) / 1024:+.1f} KiB")

del large, small, large2, compact
gc.collect()
print(f"after releasing everything      -> /dev/shm {shm_used() / MB:.1f} MiB")

print("\n-- share_memory_() breaks NumPy aliasing, because it swaps the storage --")
a = np.arange(10, dtype=np.float32)
t = torch.from_numpy(a)
print(f"torch.from_numpy(a)   aliases a: {aliases_numpy(t, a)}")
t.share_memory_()
print(f"after share_memory_() aliases a: {aliases_numpy(t, a)}   <- t moved, a did not")

shared = torch.empty(10).share_memory_()      # share first...
view = shared.numpy()                          # ...then view it from NumPy
shared[0] = 42.0
print(f"\nsafe order: torch.empty(10).share_memory_() then .numpy() -> "
      f"numpy sees {view[0]}, aliases={aliases_numpy(shared, view)}")
del t, shared, view
_ = gc.collect()

small.numel()=10 but small.untyped_storage().nbytes()=64 MiB
small.share_memory_()          -> /dev/shm +64.0 MiB
large2[:10].clone() then share  -> /dev/shm +4.0 KiB
after releasing everything      -> /dev/shm 0.2 MiB

-- share_memory_() breaks NumPy aliasing, because it swaps the storage --
torch.from_numpy(a)   aliases a: True
after share_memory_() aliases a: False   <- t moved, a did not

safe order: torch.empty(10).share_memory_() then .numpy() -> numpy sees 42.0, aliases=True


On POSIX systems, PyTorch’s shared-memory allocator uses approximately:

```c
fd = shm_open(name, O_RDWR | O_CREAT, 0600);
ftruncate(fd, size);
ptr = mmap(NULL, size, PROT_READ | PROT_WRITE, MAP_SHARED, fd, 0);
```

The current implementation is in [`ATen/MapAllocator.cpp`](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/MapAllocator.cpp).

On Linux, POSIX shared-memory objects are normally backed by the tmpfs mounted at `/dev/shm`. This means:

* The allocation consumes `/dev/shm` capacity.
* It is memory-backed, although pages can still participate in normal virtual-memory behavior.
* All processes mapping it with `MAP_SHARED` see the same physical pages.
* Changes are visible across processes, but PyTorch provides no automatic locking or synchronization.

### Two sharing strategies

`torch.multiprocessing` can hand a storage over in two ways, selectable with
`mp.set_sharing_strategy(...)`:

**`file_descriptor`** (the default on Linux, and what this notebook runs):

1. create a shared-memory object with `shm_open`,
2. `mmap` it,
3. usually `shm_unlink` the name immediately,
4. keep the FD open,
5. pass duplicated FDs to other processes (`multiprocessing.reduction.DupFd`),
6. the object dies when the last mapping/FD is released.

Because the name is unlinked in step 3, the segment shows up as `(deleted)` in `/proc/PID/maps` and
does *not* appear in `ls /dev/shm` -- you will see exactly that in the diagnostics section. Robust
against leftovers after a crash, but it consumes file descriptors (the classic
"received 0 items of ancestor" / "too many open files" DataLoader failure).

**`file_system`** sends the shared-memory object's *name* instead of a descriptor:

* fewer persistent FDs,
* but the name cannot be unlinked immediately, so a crash can leave segments behind in `/dev/shm`,
* PyTorch runs a `torch_shm_manager` helper process to clean them up after all users exit.

Use `file_system` mainly when FD limits are the actual bottleneck. The trade-off is documented in
[the multiprocessing notes](https://docs.pytorch.org/docs/main/multiprocessing.html); the
implementation is the `get_sharing_strategy()` branch of `reduce_storage()` shown below.

**The `share_memory_()` call chain**, top to bottom:

| Level | Symbol | Source (installed 2.13.0) |
| --- | --- | --- |
| Python | `Tensor.share_memory_` | `torch/_tensor.py:836` |
| Python | `TypedStorage._share_memory_` / `UntypedStorage.share_memory_` | `torch/storage.py:391`, `:1195` |
| Python | `_share_fd_cpu_` (binding) | `torch/storage.py:128` |
| C++ | `THPStorage_shareFd` | [`torch/csrc/StorageSharing.cpp`](https://github.com/pytorch/pytorch/blob/main/torch/csrc/StorageSharing.cpp) |
| C++ | `shm_open` + `ftruncate` + `mmap` | [`aten/src/ATen/MapAllocator.cpp`](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/MapAllocator.cpp) |

In [9]:
# Read the Python end of that chain in the installed build.
show_source(torch.Tensor.share_memory_)
print("\n" + "=" * 100 + "\n")
show_source(torch.storage.TypedStorage._share_memory_)
print("\n" + "=" * 100 + "\n")
show_source(torch.UntypedStorage.share_memory_)
print("\n# _share_fd_cpu_ is a C binding -- the interesting part is on the other side of it:")
print(f"# torch.UntypedStorage._share_fd_cpu_ -> {torch.UntypedStorage._share_fd_cpu_}")

# torch/_tensor.py:836   Tensor.share_memory_
    def share_memory_(self):
        if has_torch_function_unary(self):
            return handle_torch_function(Tensor.share_memory_, (self,), self)
        self._typed_storage()._share_memory_()
        return self


# torch/storage.py:1201   TypedStorage._share_memory_
    def _share_memory_(self):
        self._untyped_storage.share_memory_()
        return self


# torch/storage.py:490   UntypedStorage.share_memory_
    @_share_memory_lock_protected
    def share_memory_(self, *args, **kwargs):
        return super().share_memory_(*args, **kwargs)

# _share_fd_cpu_ is a C binding -- the interesting part is on the other side of it:
# torch.UntypedStorage._share_fd_cpu_ -> <function UntypedStorage._share_fd_cpu_ at 0xfae31dd34d60>


### Queue

`torch.multiprocessing.Queue` is a specialized wrapper around Python's native inter-process communication queue designed to move tensor data efficiently into shared memory.

The complete CPU path when passing tensor via `Queue`:

```mermaid
sequenceDiagram
    participant P as Producer
    participant F as Queue feeder
    participant K as Kernel shared memory
    participant C as Consumer

    P->>F: queue.put(tensor)
    F->>F: reduce_tensor()
    F->>K: copy to SHM if not shared
    F->>C: send FD + shape/stride/offset
    C->>K: mmap same SHM object
    C->>C: rebuild Tensor view
```

The receiver creates a new `TensorImpl` and `StorageImpl`, but its storage maps the same physical shared-memory pages. Its `data_ptr()` will usually be a different virtual address because each process has its own virtual address space.

PyTorch registers custom reducers with Python’s `ForkingPickler` for tensors and storages. The registrations can be seen at the bottom of [`torch/multiprocessing/reductions.py`](https://github.com/pytorch/pytorch/blob/main/torch/multiprocessing/reductions.py).

**Step 1: `Queue.put()` does not serialize immediately.**
A `multiprocessing.Queue` appends the object to an in-process `deque` and wakes a background
`QueueFeederThread`, which later calls `ForkingPickler.dumps()` and writes the result to the pipe
(CPython's [`Lib/multiprocessing/queues.py`](https://github.com/python/cpython/blob/main/Lib/multiprocessing/queues.py)).
Consequences worth remembering:

* `queue.put(tensor)` can return *before* PyTorch has prepared the shared storage.
* Do not mutate or reuse the tensor right after `put()` without synchronizing.
* Serialization errors surface in the feeder thread, not at the `put()` call site.
* `SimpleQueue` skips the feeder thread but uses the same reducers.

**Step 2: `reduce_tensor()`** (`torch/multiprocessing/reductions.py:221`) refuses non-leaf tensors
that require grad (autograd graphs do not cross processes -- `detach()` first), then returns, for a
dense CPU tensor, roughly:

```python
rebuild_tensor, (type(tensor), storage, (tensor.storage_offset(),
                                         tensor.size(),
                                         tensor.stride(),
                                         tensor.requires_grad))
```

**Step 3: `reduce_storage()`** (`:590`) reduces that storage separately. Under `file_descriptor`:

```python
fd, size = storage._share_fd_cpu_()
df = multiprocessing.reduction.DupFd(fd)
return rebuild_storage_fd, (type(storage), df, size)
```

Only a descriptor, a size, and the small metadata tuple go through the pipe.

**Step 4: the receiver** runs `rebuild_storage_fd()` (`:537`): detach the FD, use
`(st_ino, st_dev)` from `fstat()` as a cache key (`fd_id()`, `:522`), `_new_shared_fd_cpu(fd, size)`
to `mmap` it, then `rebuild_tensor()` (`:108`) restores size/stride/offset. The cache matters when
several views of one storage arrive: they are rebuilt as views of a *single* receiver-side storage
instead of independent mappings.

The reducers are wired into `ForkingPickler` by `init_reductions()` (`:623`), which is called when
`torch.multiprocessing` is imported.

In [10]:
import torch.multiprocessing.reductions as reductions

show_source(reductions.reduce_tensor, head=12)      # the guard clause, then the CUDA/meta/CPU split
print("\n" + "=" * 100 + "\n")
show_source(reductions.reduce_storage)
print("\n" + "=" * 100 + "\n")
show_source(reductions.fd_id)
print()
show_source(reductions.rebuild_storage_fd)
print("\n" + "=" * 100 + "\n")
show_source(reductions.rebuild_tensor)
print("\n" + "=" * 100 + "\n")
show_source(reductions.init_reductions, head=14)

# torch/multiprocessing/reductions.py:221   reduce_tensor
def reduce_tensor(tensor):
    if tensor.requires_grad and not tensor.is_leaf:
        raise RuntimeError(
            "Cowardly refusing to serialize non-leaf tensor which requires_grad, "
            "since autograd does not support crossing process boundaries.  "
            "If you just want to transfer the data, call detach() on the tensor "
            "before serializing (e.g., putting it on the queue)."
        )

    torch.utils.hooks.warn_if_has_hooks(tensor)

    # Note [CUDA IPC and the caching allocator]
    ...


# torch/multiprocessing/reductions.py:590   reduce_storage
def reduce_storage(storage):
    from . import get_sharing_strategy

    if storage.is_cuda:
        raise RuntimeError(
            "Cannot pickle CUDA storage; try pickling a CUDA tensor instead"
        )
    elif storage.device.type == "meta":
        raise RuntimeError(
            "Cannot pickle meta storage; try pickling a meta tensor instea

In [11]:
# How big is the message that actually travels through the pipe?
import io
import pickle
from multiprocessing.reduction import ForkingPickler


def forking_pickled_size(t):
    buf = io.BytesIO()
    ForkingPickler(buf, pickle.HIGHEST_PROTOCOL).dump(t)
    return len(buf.getvalue())


shm_before = shm_used()
small = torch.zeros(1024, dtype=torch.float32).share_memory_()             # 4 KiB
big = torch.zeros(10 * MB // 4, dtype=torch.float32).share_memory_()       # 10 MiB

print(f"{'tensor':16s} {'payload bytes':>14s} {'ForkingPickler':>15s} {'plain pickle':>14s}")
for name, t in [("4 KiB tensor", small), ("10 MiB tensor", big)]:
    print(f"{name:16s} {t.untyped_storage().nbytes():>14,d} "
          f"{forking_pickled_size(t):>15,d} {len(pickle.dumps(t)):>14,d}")

print("\nForkingPickler payload is ~constant: it is (fd, size, shape, stride, offset, dtype).")
print("Plain pickle grows with the tensor -- that is what torch.save/torch.load do, and what an")
print("ordinary numpy array sent through a Queue does too.")

del small, big
gc.collect()
from multiprocessing import resource_sharer
print(f"\n/dev/shm still holding {(shm_used() - shm_before) / MB:+.1f} MiB after deleting both "
      f"tensors,\nbecause DupFd registered {len(resource_sharer._resource_sharer._cache)} descriptor(s) "
      f"with multiprocessing's\nresource sharer and nobody ever consumed them -- a reduced-but-never-sent "
      f"tensor keeps its\nshared segment alive. In a real queue the receiver detaches the FD and this "
      f"resolves itself.")

tensor            payload bytes  ForkingPickler   plain pickle
4 KiB tensor              4,096             338          4,485
10 MiB tensor        10,485,760             342     10,486,162

ForkingPickler payload is ~constant: it is (fd, size, shape, stride, offset, dtype).
Plain pickle grows with the tensor -- that is what torch.save/torch.load do, and what an
ordinary numpy array sent through a Queue does too.



/dev/shm still holding +10.0 MiB after deleting both tensors,
because DupFd registered 2 descriptor(s) with multiprocessing's
resource sharer and nobody ever consumed them -- a reduced-but-never-sent tensor keeps its
shared segment alive. In a real queue the receiver detaches the FD and this resolves itself.


**Experiment: the round trip.** The child below receives the tensor, reports its own `data_ptr()`,
and writes into it. Two things to watch: the addresses differ (each process has its own virtual
address space) while the *physical* pages are the same, so the parent observes the child's write.

In [12]:
def child_mutates(inq, outq):
    t = inq.get()                       # rebuild_storage_fd + rebuild_tensor happen here
    outq.put({"pid": os.getpid(), "data_ptr": t.data_ptr(), "is_shared": t.is_shared(),
              "shape": tuple(t.shape), "storage_nbytes": t.untyped_storage().nbytes(),
              "value_seen": t[0].item()})
    t.add_(100.0)                       # write straight into the shared pages
    outq.put("done")


ctx = mp.get_context("fork")            # fork: the worker above can stay in this notebook
inq, outq = ctx.Queue(), ctx.Queue()

t = torch.zeros(4, dtype=torch.float32).share_memory_()
print(f"parent pid={os.getpid()} data_ptr=0x{t.data_ptr():x} values={t.tolist()}")

p = ctx.Process(target=child_mutates, args=(inq, outq))
p.start()
inq.put(t)
info = outq.get()
outq.get()
p.join()

print(f"child  pid={info['pid']} data_ptr=0x{info['data_ptr']:x} is_shared={info['is_shared']} "
      f"shape={info['shape']} storage_nbytes={info['storage_nbytes']} value_seen={info['value_seen']}")
print(f"same virtual address? {info['data_ptr'] == t.data_ptr()}  "
      f"<- different mapping, same physical pages")
print(f"parent sees after child's add_(100): {t.tolist()}")
assert t[0].item() == 100.0
del t
_ = gc.collect()

parent pid=810933 data_ptr=0xfae432831000 values=[0.0, 0.0, 0.0, 0.0]


child  pid=810995 data_ptr=0xfae4327ef000 is_shared=True shape=(4,) storage_nbytes=16 value_seen=0.0
same virtual address? False  <- different mapping, same physical pages
parent sees after child's add_(100): [100.0, 100.0, 100.0, 100.0]


**Experiment: what the first send costs.** Same tensor, same queue, three sends. The only variable
is where the storage lived beforehand. Watch both columns: the wall time and the `/dev/shm` delta
move together, because they are the same event -- one 256 MiB `memcpy` into a fresh shared mapping.

In [13]:
def echo_len(inq, outq):
    t = inq.get()
    outq.put(t.shape[0])                # confirms the child has rebuilt the tensor


def timed_send(t, label):
    q_in, q_out = ctx.Queue(), ctx.Queue()
    p = ctx.Process(target=echo_len, args=(q_in, q_out))
    p.start()
    shm_before, t0 = shm_used(), time.perf_counter()
    q_in.put(t)
    q_out.get()
    dt = (time.perf_counter() - t0) * 1e3
    delta = (shm_used() - shm_before) / MB
    p.join()
    print(f"{label:40s} {dt:8.2f} ms   /dev/shm {delta:+7.1f} MiB   is_shared={t.is_shared()}")


nbytes = 256 * MB
plain = torch.zeros(nbytes // 4, dtype=torch.float32)
preshared = torch.zeros(nbytes // 4, dtype=torch.float32).share_memory_()   # copy paid already

print(f"sending a {nbytes // MB} MiB tensor through a Queue:")
timed_send(plain, "1st send, ordinary CPU tensor")
timed_send(plain, "2nd send, same tensor (now shared)")
timed_send(preshared, "share_memory_() called beforehand")

del plain, preshared
gc.collect()
print(f"\n/dev/shm after cleanup: {shm_used() / MB:.1f} MiB")

sending a 256 MiB tensor through a Queue:


1st send, ordinary CPU tensor               57.19 ms   /dev/shm  +256.0 MiB   is_shared=True


2nd send, same tensor (now shared)           3.48 ms   /dev/shm    +0.0 MiB   is_shared=True


share_memory_() called beforehand            3.03 ms   /dev/shm    +0.0 MiB   is_shared=True



/dev/shm after cleanup: 10.3 MiB


### Fork is not the same as shared memory

With a fork-like start method the child initially inherits the parent's page tables, which is why
"the child can already see my tensors" feels like sharing. It is not:

* ordinary anonymous memory is **copy-on-write** -- reads may hit the same physical pages, but the
  first write gives the writer a private copy,
* a storage moved to shared memory is mapped `MAP_SHARED`, so writes stay visible in both processes.

With `spawn` the child starts a fresh interpreter: arguments and the `Dataset` are pickled, and
tensor arguments go through the same reducers as above (plain Python objects are pickled normally).

The cell below calls `os.fork()` directly, so Python emits
`DeprecationWarning: This process is multi-threaded, use of fork() may lead to deadlocks in the child`
-- a Jupyter kernel always has helper threads, and a lock held by another thread at fork time stays
locked forever in the child. Here the child only writes into two tensors and calls `os._exit(0)`, so
there is nothing to deadlock on. That warning is the same reason CUDA and threaded runtimes push you
toward `spawn`.

In [14]:
# Same fill_() in the child, two different storages. Only one write survives.
plain = torch.zeros(4)
shared = torch.zeros(4).share_memory_()

pid = os.fork()
if pid == 0:                 # child: inherits the parent's page tables
    plain.fill_(7.0)         # anonymous memory -> copy-on-write -> private copy
    shared.fill_(7.0)        # MAP_SHARED -> same physical pages
    os._exit(0)              # _exit, not exit: never unwind a forked notebook kernel

os.waitpid(pid, 0)
print(f"plain  after child's fill_(7): {plain.tolist()}   <- copy-on-write, parent unchanged")
print(f"shared after child's fill_(7): {shared.tolist()}   <- MAP_SHARED, write is visible")
assert plain.sum().item() == 0.0 and shared.sum().item() == 28.0
del plain, shared
_ = gc.collect()

plain  after child's fill_(7): [0.0, 0.0, 0.0, 0.0]   <- copy-on-write, parent unchanged
shared after child's fill_(7): [7.0, 7.0, 7.0, 7.0]   <- MAP_SHARED, write is visible


/tmp/ipykernel_810933/1896710216.py:5: DeprecationWarning: This process (pid=810933) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### Pytorch Multi-process Dataloader

With `num_workers > 0`, the classic DataLoader path is:

```text
main process
    │ indices
    ▼
per-worker multiprocessing index queues
    │
    ▼
worker process
    dataset.__getitem__ / __getitems__
    collate_fn
    │ batch
    ▼
worker_result_queue
    │
    ├─ pin_memory=False ─────────────► user
    │
    └─ pin_memory=True
          pin-memory thread
          local queue
              └──────────────────────► user
```

The topology lives in
[`dataloader.py`](https://github.com/pytorch/pytorch/blob/main/torch/utils/data/dataloader.py); the
worker calls `fetcher.fetch(index)` and then `data_queue.put((idx, data))` in
[`worker.py`](https://github.com/pytorch/pytorch/blob/main/torch/utils/data/_utils/worker.py)
(`:374` and `:392` in the installed build).

**Why default collation is efficient.** `collate_tensor_fn` checks whether it is running inside a
worker, and if so allocates the stacked batch *in shared memory* before stacking into it
(`torch/utils/data/_utils/collate.py:269-275`):

```python
if torch.utils.data.get_worker_info() is not None:
    numel = sum(x.numel() for x in batch)
    storage = elem._typed_storage()._new_shared(numel, device=elem.device)
    out = elem.new(storage).resize_(len(batch), *list(elem.size()))
return torch.stack(batch, 0, out=out)
```

So the batch is *born* shared, and `reduce_storage()` finds an already-shared storage with nothing to
copy. The per-sample copy into the stacked batch still happens -- that is batching, not IPC.

```text
without pinning:  decoded sample memory -> stack/collate copy -> shared batch
                                                                   └─ mmap in main process: no payload copy

with pin_memory:  decoded sample memory -> stack/collate -> shared pageable batch
                                                              └─ pin copy -> pinned batch
                                                                               └─ H2D copy -> CUDA
```

In [15]:
import torch.utils.data._utils.collate as collate_mod
import torch.utils.data._utils.worker as worker_mod

show_source(collate_mod.collate_tensor_fn)
print("\n" + "=" * 100 + "\n")
show_lines(worker_mod, 368, 393)

# torch/utils/data/_utils/collate.py:246   collate_tensor_fn
def collate_tensor_fn(
    batch,
    *,
    collate_fn_map: dict[type | tuple[type, ...], Callable] | None = None,
):
    elem = batch[0]
    out = None
    if elem.is_nested:
        raise RuntimeError(
            "Batches of nested tensors are not currently supported by the default collate_fn; "
            "please provide a custom collate_fn to handle them appropriately."
        )
    if elem.layout in {
        torch.sparse_coo,
        torch.sparse_csr,
        torch.sparse_bsr,
        torch.sparse_csc,
        torch.sparse_bsc,
    }:
        raise RuntimeError(
            "Batches of sparse tensors are not currently supported by the default collate_fn; "
            "please provide a custom collate_fn to handle them appropriately."
        )
    if torch.utils.data.get_worker_info() is not None:
        # If we're in a background process, concatenate directly into a
        # shared memory tensor to avoid an extra

**Experiment.** Three configurations, three observable states of the same batch. `is_shared()` tells
you whether the bytes are in `/dev/shm`; `is_pinned()` tells you whether they are in
`cudaHostAlloc`'d memory. Note that they are mutually exclusive here: pinning is another copy, into a
different kind of allocation.

The `/dev/shm` column is the sizing rule made visible -- we hold four batches, and the workers keep
prefetching ahead of us.

In [16]:
from torch.utils.data import DataLoader, Dataset


class Synthetic(Dataset):
    # each sample is 1 MiB of float32, so /dev/shm movement is easy to see
    def __init__(self, n=32, numel=256 * 1024):
        self.n, self.numel = n, numel

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        return torch.full((self.numel,), float(i))


ds = Synthetic()
sample_mib = ds.numel * 4 / MB
print(f"sample = {sample_mib:.0f} MiB, batch_size=4 -> batch = {4 * sample_mib:.0f} MiB\n")
print(f"{'config':44s} {'is_shared':>10s} {'is_pinned':>10s} {'/dev/shm held':>16s}")
print(f"{'baseline (no loader)':44s} {'-':>10s} {'-':>10s} {shm_used() / MB:>13.0f} MiB")


def probe(label, **kwargs):
    loader = DataLoader(ds, batch_size=4, **kwargs)
    it = iter(loader)
    held = [next(it) for _ in range(4)]       # keep 4 batches alive on purpose
    batch = held[0]
    print(f"{label:44s} {str(batch.is_shared()):>10s} {str(batch.is_pinned()):>10s} "
          f"{shm_used() / MB:>13.0f} MiB")
    del held, it, loader
    gc.collect()


probe("num_workers=0", num_workers=0)
probe("num_workers=2, prefetch_factor=2", num_workers=2, prefetch_factor=2)
probe("num_workers=2, prefetch_factor=4", num_workers=2, prefetch_factor=4)
# pin_memory=True initializes CUDA in this process, so keep it last: no fork()ing after this point.
probe("num_workers=2, pin_memory=True", num_workers=2, pin_memory=True)
print(f"\nafter releasing every batch: /dev/shm = {shm_used() / MB:.0f} MiB")

sample = 1 MiB, batch_size=4 -> batch = 4 MiB

config                                        is_shared  is_pinned    /dev/shm held
baseline (no loader)                                  -          -            10 MiB
num_workers=0                                     False      False            10 MiB


num_workers=2, prefetch_factor=2                   True      False            38 MiB


num_workers=2, prefetch_factor=4                   True      False            35 MiB


num_workers=2, pin_memory=True                    False       True            18 MiB

after releasing every batch: /dev/shm = 10 MiB


Reading the numbers above:

* `num_workers=0` -- collation happens in this process, `get_worker_info()` is `None`, so the batch
  is an ordinary CPU allocation. Nothing enters `/dev/shm`.
* `num_workers=2` -- the batch arrives already shared, and raising `prefetch_factor` raises the
  `/dev/shm` high-water mark, because more batches are in flight at once.
* `pin_memory=True` -- the batch you receive is pinned and *not* shared: the pin-memory thread copied
  it out of the shared segment into pinned host memory, which is why less `/dev/shm` is held.

**Sizing `/dev/shm` for a DataLoader.** A first-order estimate:

```text
SHM demand ≈ batch bytes x num_workers x prefetch_factor
```

then add room for:

* batches waiting for in-order delivery,
* batches still retained by your code,
* multiple tensors per batch (inputs, labels, masks...),
* allocator and page alignment overhead.

If you append every returned batch to a list, those mappings stay alive and `/dev/shm` grows for the
whole epoch -- the classic `Bus error / ENOSPC` in a container with the default 64 MiB `/dev/shm`.

**Try it:** raise `num_workers` or `prefetch_factor` and watch the `/dev/shm` column; or return a
tuple of two tensors from `__getitem__` and watch it roughly double.

### GPU IPC transfer GPU tensor between processes

A CUDA tensor's payload never goes near `/dev/shm`. `reduce_tensor()` takes the
`storage._share_cuda_()` branch (`reductions.py:340`) and serializes:

* the CUDA IPC memory handle,
* the allocation size,
* the storage's offset *inside the caching-allocator block*,
* the tensor's offset, shape and strides,
* IPC reference-counter information,
* optional CUDA event synchronization info,

and the receiver runs `rebuild_cuda_tensor()` (`:153`), which opens the handle and builds a storage
pointing into the same GPU allocation. The long comment above `reduce_tensor` ("Note [CUDA IPC and
the caching allocator]") explains why the *whole* `cudaMalloc` block has to be sent rather than just
your storage, and why the resulting tensor must not be resizable. The C++ side is
`THPStorage_shareCuda` in
[`torch/csrc/StorageSharing.cpp`](https://github.com/pytorch/pytorch/blob/main/torch/csrc/StorageSharing.cpp).

Restrictions that actually bite:

* use `spawn` or `forkserver`, never `fork`, with CUDA,
* the producer must stay alive while consumers hold the tensor,
* abnormal consumer termination can keep the allocation alive,
* do not forward a *received* IPC tensor to a third process without cloning it,
* for DataLoader, return CPU tensors and use `pin_memory=True` instead of returning CUDA tensors.

Because CUDA IPC needs `spawn`, this one cannot run inline in a fork-based notebook: the cell below
writes a small script and runs it as a subprocess. Expect the parent's teardown warning
`Producer process has been terminated before all shared CUDA tensors released` -- that is restriction
number two announcing itself, not a failure of the demo.

In [17]:
IPC_DEMO = r'''
import os
import torch
import torch.multiprocessing as mp


def child(q, done):
    t = q.get()                      # opens the CUDA IPC handle in this process
    print(f"[child  pid={os.getpid()}] received {tuple(t.shape)} on {t.device} "
          f"data_ptr=0x{t.data_ptr():x} values={t.tolist()}", flush=True)
    t.add_(100)                      # writes into the parent's GPU allocation
    torch.cuda.synchronize()
    done.put(True)


if __name__ == "__main__":
    ctx = mp.get_context("spawn")    # CUDA IPC requires spawn/forkserver, never fork
    q, done = ctx.Queue(), ctx.Queue()
    t = torch.zeros(4, device="cuda")
    print(f"[parent pid={os.getpid()}] created  {tuple(t.shape)} on {t.device} "
          f"data_ptr=0x{t.data_ptr():x} values={t.tolist()}", flush=True)
    p = ctx.Process(target=child, args=(q, done))
    p.start()
    q.put(t)
    done.get()
    p.join()
    torch.cuda.synchronize()
    print(f"[parent pid={os.getpid()}] after child's add_(100): {t.tolist()}", flush=True)
'''

if torch.cuda.is_available():
    import shutil
    tmp = tempfile.mkdtemp(prefix="tensor-travel-ipc-")
    script = pathlib.Path(tmp) / "cuda_ipc_demo.py"
    script.write_text(IPC_DEMO)
    proc = subprocess.run([sys.executable, str(script)], capture_output=True, text=True, timeout=600)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print("--- stderr ---")
        print(proc.stderr.strip()[-500:])
    print(f"\nexit code: {proc.returncode}")
    shutil.rmtree(tmp)
else:
    print("no CUDA device available -- skipping the CUDA IPC demo")

[parent pid=811055] created  (4,) on cuda:0 data_ptr=0x32ee00000 values=[0.0, 0.0, 0.0, 0.0]
[child  pid=811093] received (4,) on cuda:0 data_ptr=0x32ee00000 values=[0.0, 0.0, 0.0, 0.0]
[parent pid=811055] after child's add_(100): [100.0, 100.0, 100.0, 100.0]
--- stderr ---
[W728 23:39:56.373922472 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]

exit code: 0


Note the detail in the output above: on this machine the child reports the *same* `data_ptr` as the
parent, unlike the CPU case where the addresses differed. CUDA uses a unified virtual address space
per device, so an IPC-mapped allocation can land at the same device address in both processes. Do not
depend on it -- depend on the handle.

### Transfer Between Programming Language Boundary

*Left open for now -- the reference notes this notebook was assembled from do not cover it yet.*

Sketch of what belongs here:

- DLPack as the lingua franca: `__dlpack__` / `__dlpack_device__`, `torch.utils.dlpack.to_dlpack`,
  `torch.from_dlpack` -- who owns the capsule, and when the exporter is allowed to free it.
- tvm-ffi's tensor ABI: <https://tvm.apache.org/ffi/concepts/tensor.html>
- The same `at::from_blob` trick as the NumPy section, but with a foreign runtime holding the
  allocation (JAX/CuPy/TensorFlow, or a plain C++ library).
- Stream semantics across the boundary: a DLPack export from CUDA carries a stream, and ignoring it
  is a data race, not a slowdown.

## Tensor Transfer Between Device Boundary
When tensor pass device doundary, we have to copy tensor storage, we want to understand how tensor data is transfered 

### From Disk to MEM

An "mmap tensor" is not a separate tensor type. It is a regular tensor whose `Storage` points into a
virtual-memory mapping instead of an ordinary allocator allocation -- the same mechanism the
process-boundary section used, pointed at a different backing object:

| Mapping | Backing object | Typical purpose |
| --- | --- | --- |
| PyTorch multiprocessing SHM | POSIX shared-memory object | IPC between local processes |
| `torch.from_file()` | user-specified file | persistent or private file-backed tensor |
| `torch.load(..., mmap=True)` | PyTorch checkpoint file | lazy, demand-paged checkpoint loading |
| `numpy.memmap` | user-specified file | NumPy file-backed array |

`torch.from_file()` picks the mapping flavor:

* `shared=True` -> `MAP_SHARED`: writes are visible to other mappings and update the file,
* `shared=False` -> `MAP_PRIVATE`: writes are copy-on-write and never reach the file.

Only CPU tensors can be file-mapped. `torch.load(path, mmap=True)` maps each checkpoint storage
instead of eagerly reading it into a fresh CPU allocation; pages arrive on demand through page
faults. It saves the eager copy and the peak memory, but touching every page still reads all the data
eventually.

Source pointers: the `mmap` branch of `torch.serialization.load` (`torch/serialization.py:1584-1599`,
which routes through `torch.UntypedStorage.from_file`, `torch/storage.py:211`), and
`get_default_mmap_options` / `set_default_mmap_options` (`:200`, `:229`) which choose
`MAP_PRIVATE` (default) or `MAP_SHARED`. Below that it is `MapAllocator` again.

In [18]:
import shutil

LOAD_PROBE = r'''
import json
import os
import pathlib
import sys
import time

import torch

MB = 1024 * 1024
ckpt, mmap_flag = sys.argv[1], sys.argv[2] == "mmap"


def rss_mib():
    fields = pathlib.Path("/proc/self/statm").read_text().split()
    return int(fields[1]) * os.sysconf("SC_PAGE_SIZE") / MB


rss0, t0 = rss_mib(), time.perf_counter()
sd = torch.load(ckpt, weights_only=True, mmap=mmap_flag)
load_ms, rss1 = (time.perf_counter() - t0) * 1e3, rss_mib()
t1 = time.perf_counter()
checksum = sum(float(v.sum()) for v in sd.values())          # touch every page
touch_ms, rss2 = (time.perf_counter() - t1) * 1e3, rss_mib()
print(json.dumps({"load_ms": load_ms, "rss_after_load": rss1 - rss0,
                  "touch_ms": touch_ms, "rss_after_touch": rss2 - rss1,
                  "mib": sum(v.numel() * v.element_size() for v in sd.values()) / MB}))
'''

tmpdir = pathlib.Path(tempfile.mkdtemp(prefix="tensor-travel-"))
ckpt = tmpdir / "checkpoint.pt"
probe = tmpdir / "load_probe.py"
probe.write_text(LOAD_PROBE)
state = {f"layer{i}.weight": torch.randn(64, 1024, 256) for i in range(2)}      # 2 x 64 MiB
torch.save(state, ckpt)
del state
_ = gc.collect()
print(f"checkpoint on disk: {ckpt.stat().st_size / MB:.0f} MiB")
print("Each row runs in a fresh interpreter -- RSS in this long-lived kernel would be meaningless,")
print("since the allocator already holds enough resident pages to absorb a 128 MiB copy.")
print("The eager row runs first, so both see a warm page cache: this compares copying to mapping,")
print("not disk speed.\n")

for label, mode in [("torch.load(...)", "eager"), ("torch.load(..., mmap=True)", "mmap")]:
    out = subprocess.run([sys.executable, str(probe), str(ckpt), mode],
                         capture_output=True, text=True, timeout=600)
    r = json.loads(out.stdout)
    print(f"{label:28s} load {r['load_ms']:7.1f} ms  RSS {r['rss_after_load']:+7.1f} MiB | "
          f"touching all {r['mib']:.0f} MiB: {r['touch_ms']:6.1f} ms  "
          f"RSS {r['rss_after_touch']:+7.1f} MiB")

print("\n-- torch.from_file: the same machinery aimed at your own file --")
raw = tmpdir / "raw.bin"
shared_t = torch.from_file(str(raw), shared=True, size=8, dtype=torch.float32)
shared_t.fill_(3.0)
print(f"shared=True  : file is now {raw.stat().st_size} bytes, "
      f"re-reading it gives {torch.from_file(str(raw), shared=True, size=8, dtype=torch.float32)[:4].tolist()}")

private_t = torch.from_file(str(raw), shared=False, size=8, dtype=torch.float32)
private_t.fill_(9.0)
reread = torch.from_file(str(raw), shared=True, size=8, dtype=torch.float32)
print(f"shared=False : this view reads {private_t[:4].tolist()} but the file still holds "
      f"{reread[:4].tolist()}   <- MAP_PRIVATE, copy-on-write")

del shared_t, private_t, reread
gc.collect()
shutil.rmtree(tmpdir)

checkpoint on disk: 128 MiB
Each row runs in a fresh interpreter -- RSS in this long-lived kernel would be meaningless,
since the allocator already holds enough resident pages to absorb a 128 MiB copy.
The eager row runs first, so both see a warm page cache: this compares copying to mapping,
not disk speed.



torch.load(...)              load    37.0 ms  RSS  +129.1 MiB | touching all 128 MiB:    2.5 ms  RSS    +2.6 MiB


torch.load(..., mmap=True)   load     0.9 ms  RSS    +1.0 MiB | touching all 128 MiB:    3.0 ms  RSS  +130.6 MiB

-- torch.from_file: the same machinery aimed at your own file --
shared=True  : file is now 32 bytes, re-reading it gives [3.0, 3.0, 3.0, 3.0]
shared=False : this view reads [9.0, 9.0, 9.0, 9.0] but the file still holds [3.0, 3.0, 3.0, 3.0]   <- MAP_PRIVATE, copy-on-write


### From CPU MEM to GPU

Crossing to the GPU is the one boundary where a payload copy is unavoidable -- the question is only
how many copies, and whether the CPU has to wait for them.

The DMA engine can only read from memory the OS has promised not to move: **pinned** (page-locked)
host memory. So a pageable source tensor forces the runtime to stage through an internal pinned
buffer and, in practice, to synchronize. `Tensor.pin_memory()` makes that staging explicit and
reusable:

```text
pageable CPU tensor ──pin_memory() copy──► pinned CPU tensor ──cudaMemcpyAsync──► CUDA tensor
```

Pinned memory is *not* the `/dev/shm` allocation from the process-boundary section -- it comes from
`cudaHostAlloc` via PyTorch's `CachingHostAllocator`, and a DataLoader batch that gets pinned is
therefore copied out of shared memory (exactly what the `is_shared`/`is_pinned` table showed).

Source pointers: `Tensor.pin_memory` -> `at::native::_pin_memory` ->
[`aten/src/ATen/cuda/CachingHostAllocator.cpp`](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/cuda/CachingHostAllocator.cpp),
and the `non_blocking` decision in
[`aten/src/ATen/native/cuda/Copy.cu`](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/cuda/Copy.cu)
(`copy_device_to_device` / `copy_from_host_to_device`), documented in the
[pinned-memory tutorial](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html).

**Read the second table, not the first, on this machine.** This box is a GB10 (Grace Blackwell
superchip) where host and device memory are physically the same LPDDR behind a coherent fabric, so
pinned and pageable reach nearly identical bandwidth. On a discrete PCIe GPU the pinned row is
typically 1.5-2x faster. What does *not* change with the platform is the asynchrony: only a pinned
source lets `copy_(..., non_blocking=True)` return before the transfer finishes.

In [19]:
if not torch.cuda.is_available():
    print("no CUDA device -- skipping the host-to-device measurements")
else:
    print(f"device: {torch.cuda.get_device_name(0)}")
    nbytes = 256 * MB
    pageable = torch.empty(nbytes // 4, dtype=torch.float32)
    pinned = pageable.pin_memory()          # this call itself is a full CPU->CPU copy
    dst = torch.empty(nbytes // 4, dtype=torch.float32, device="cuda")

    print(f"pageable : is_pinned={pageable.is_pinned()} is_shared={pageable.is_shared()}")
    print(f"pinned   : is_pinned={pinned.is_pinned()} is_shared={pinned.is_shared()} "
          f"(cudaHostAlloc memory, not /dev/shm; different address: "
          f"{pinned.data_ptr() != pageable.data_ptr()})\n")

    def bench(src, non_blocking, iters=5):
        dst.copy_(src, non_blocking=non_blocking)
        torch.cuda.synchronize()
        start, end = torch.cuda.Event(True), torch.cuda.Event(True)
        start.record()
        for _ in range(iters):
            dst.copy_(src, non_blocking=non_blocking)
        end.record()
        torch.cuda.synchronize()
        ms = start.elapsed_time(end) / iters
        return ms, nbytes / MB / 1024 / (ms / 1e3)

    print(f"{'H2D copy of 256 MiB':34s} {'per copy':>10s} {'bandwidth':>12s}")
    for label, src, nb in [("pageable, non_blocking=False", pageable, False),
                           ("pageable, non_blocking=True", pageable, True),
                           ("pinned,   non_blocking=False", pinned, False),
                           ("pinned,   non_blocking=True", pinned, True)]:
        ms, gibs = bench(src, nb)
        print(f"{label:34s} {ms:7.2f} ms {gibs:9.2f} GiB/s")

    print("\nwhen does copy_(non_blocking=True) actually return early?")
    for label, src in [("pageable", pageable), ("pinned", pinned)]:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        dst.copy_(src, non_blocking=True)
        returned = (time.perf_counter() - t0) * 1e3
        torch.cuda.synchronize()
        total = (time.perf_counter() - t0) * 1e3
        print(f"  {label:9s} copy_ returned after {returned:6.2f} ms, "
              f"transfer finished after {total:6.2f} ms")

    del pageable, pinned, dst
    gc.collect()
    torch.cuda.empty_cache()

device: NVIDIA GB10
pageable : is_pinned=False is_shared=False
pinned   : is_pinned=True is_shared=False (cudaHostAlloc memory, not /dev/shm; different address: True)

H2D copy of 256 MiB                  per copy    bandwidth
pageable, non_blocking=False          4.51 ms     55.47 GiB/s
pageable, non_blocking=True           4.50 ms     55.49 GiB/s


pinned,   non_blocking=False          4.76 ms     52.53 GiB/s
pinned,   non_blocking=True           4.55 ms     54.96 GiB/s

when does copy_(non_blocking=True) actually return early?
  pageable  copy_ returned after   4.51 ms, transfer finished after   4.51 ms
  pinned    copy_ returned after   0.00 ms, transfer finished after   5.26 ms


### From GPU to GPU (NCCL)

*Left open for now -- and not measurable on this machine, which has a single GB10 GPU
(`torch.cuda.device_count() == 1`).*

Sketch of what belongs here:

- `cudaMemcpyPeer` and `torch.cuda.can_device_access_peer()`: when a D2D copy goes over NVLink /
  NVSwitch versus bouncing through host memory.
- CUDA IPC between processes on *different* devices, versus the collectives path.
- `torch.distributed` + NCCL: ring/tree algorithms, `all_reduce` / `all_gather` / `broadcast`, and
  why the buffer must be contiguous and on the right device.
- Where inference engines actually spend this budget: tensor-parallel all-reduces per layer, and
  KV-cache movement between prefill and decode workers.

## Tensor Transfer Between Nodes

*Left open for now -- single-node machine, and the reference notes do not cover this yet.*

Sketch of what belongs here:

- RDMA / GPUDirect: NIC DMA straight to GPU memory, and the registration (pinning) it requires.
- NCCL over IB/RoCE versus TCP; `NCCL_SOCKET_IFNAME`, rail-optimized topologies.
- Disaggregated inference: moving a KV cache between prefill and decode nodes, and why the wire
  format is usually a flat contiguous buffer plus a small metadata header -- the same
  storage-plus-metadata split this notebook started with, serialized.

## Practical diagnostics

The commands and attributes worth reaching for when a real workload is misbehaving. The `(deleted)`
marker in `/proc/self/maps` is the `file_descriptor` strategy doing step 3 (`shm_unlink`) -- the
segment is alive and mapped, it just has no name left, which is why `ls /dev/shm` does not show it.

When interpreting memory metrics:

* RSS counts the same shared pages in *every* process that maps them,
* so summing RSS across a DataLoader's workers substantially overstates physical memory,
* PSS divides shared pages among the processes mapping them (`/proc/PID/smaps_rollup`),
* `/dev/shm` usage measures the backing allocation independently of per-process RSS accounting.

In [20]:
t = torch.zeros(32 * MB // 4, dtype=torch.float32).share_memory_()      # 32 MiB, shared

print(f"sharing strategy : {mp.get_sharing_strategy()} "
      f"(available: {sorted(mp.get_all_sharing_strategies())})")
print(f"tensor           : is_shared={t.is_shared()} "
      f"nbytes={t.untyped_storage().nbytes() / MB:.0f} MiB "
      f"offset={t.storage_offset()} stride={t.stride()} data_ptr=0x{t.data_ptr():x}")
print(f"/dev/shm used    : {shm_used() / MB:.1f} MiB")

print("\n$ df -h /dev/shm")
print(subprocess.run(["df", "-h", "/dev/shm"], capture_output=True, text=True).stdout.rstrip())

print("\n$ ls /dev/shm      (file_descriptor unlinks the name, so torch_* segments are invisible here)")
print(sorted(os.listdir("/dev/shm")) or "[empty]")

print("\n$ grep /dev/shm /proc/self/maps")
for line in pathlib.Path("/proc/self/maps").read_text().splitlines():
    if "/dev/shm" in line:
        print(line)

print("\n$ grep -E 'Rss|Pss|Shared|Private_Dirty' /proc/self/smaps_rollup")
for line in pathlib.Path("/proc/self/smaps_rollup").read_text().splitlines():
    if line.split(":")[0] in {"Rss", "Pss", "Shared_Clean", "Shared_Dirty", "Private_Dirty"}:
        print("  " + line.strip())

print("\nother things worth checking on a real box:")
print("  lsof /dev/shm | grep torch          # who is holding the segments (or the FDs)")
print("  cat /proc/$PID/maps | grep torch_   # per-process mappings, including (deleted) ones")
print("  ulimit -n                            # file_descriptor strategy burns FDs")

del t
gc.collect()
print(f"\n/dev/shm after cleanup: {shm_used() / MB:.1f} MiB")

sharing strategy : file_descriptor (available: ['file_descriptor', 'file_system'])
tensor           : is_shared=True nbytes=32 MiB offset=0 stride=(1,) data_ptr=0xfae3d55af000
/dev/shm used    : 42.3 MiB

$ df -h /dev/shm
Filesystem      Size  Used Avail Use% Mounted on
tmpfs            60G   43M   60G   1% /dev/shm

$ ls /dev/shm      (file_descriptor unlinks the name, so torch_* segments are invisible here)
['cuda.shm.3e8.c4e1f.1', 'cuda.shm.3e8.c5a83.1', 'cuda.shm.3e8.c5e1c.1', 'cuda.shm.3e8.c602f.1']

$ grep /dev/shm /proc/self/maps
fae3d55af000-fae3d75af000 rw-s 00000000 00:1e 469                        /dev/shm/torch_810933_1491049113_11 (deleted)
fae432832000-fae432833000 rw-s 00000000 00:1e 373                        /dev/shm/sem.LTgA86 (deleted)
fae432833000-fae432834000 rw-s 00000000 00:1e 372                        /dev/shm/sem.ncF3Ze (deleted)
fae432871000-fae432872000 rw-s 00000000 00:1e 371                        /dev/shm/sem.CTR6kX (deleted)
fae432872000-fae432873000 rw-

## Summary

Everything above is one idea applied at four boundaries: **a tensor is metadata plus a storage, and
you move a tensor by moving as little as the boundary allows.**

| Boundary | What crosses | Payload copy |
| --- | --- | --- |
| view / reshape | a new `TensorImpl` | none |
| library (NumPy) | pointer + strides + a deleter (`at::from_blob`) | none, unless dtype/device/order forces it |
| process (CPU) | an FD (or a name) + shape/stride/offset | once, on the first send of an unshared storage |
| process (CUDA) | an IPC handle + allocation offsets | none |
| disk -> memory | a mapping (`mmap`) or a read | none up front with `mmap=True`; pages fault in on touch |
| host -> device | a DMA transfer | always; pinning adds one CPU copy to make it async |

So PyTorch multiprocessing is best described as **handle passing after shared-storage placement**,
not universally copy-free serialization. The DataLoader is the design lesson: it cannot avoid
materializing a batch, so it arranges for that unavoidable copy to land *directly in the final shared
buffer* -- and then the IPC really is free.

Still open in this notebook: the programming-language boundary (DLPack / tvm-ffi), GPU-to-GPU, and
node-to-node over the network.